# 21 — Rust controller bakeoff (filtered F3)

Compare ladder controllers under **filtered** beliefs via `session.act` on the
richest practical channel package:

- **GSIN** lot-level code (`code_type=gsin`)
- **Scan waste** (`scan_waste=true`)
- **Temperature history** on delivery (`delivery_history=temperature_history`)

Preset **F3** in `channels_for_preset`.

| Arm | Filtered (F3) |
| --- | --- |
| `constant` | yes |
| `sw` | yes |
| `sla_pb` | yes |

**Production budget:** `n_burn=2`, `n_score=45` → **47-day** horizon per shard.
Modal flat-spawn: **30** shards (10 seeds × 3 arms).
Per-arm tuned α from `experiments/tuned_alpha.json`; ρ: `sw`=0.80 (damped_sw BO),
`sla_pb`=0.50 (SOO BO), `constant`=0.80.


In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for _candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (_candidate / "src" / "blueberries_voi").is_dir():
        REPO_ROOT = _candidate
        break

_wheel_dir = REPO_ROOT / "dist" / "wheel"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)

from blueberries_voi.experiments.controller_bakeoff import (
    ARM_LABELS,
    DEFAULT_CONTROLLER_SEEDS,
    FILTERED_ARMS,
    FILTERED_OBS_PRESET,
    PRODUCTION_N_SCORE,
    filtered_obs_channels,
    resolve_arm_alpha,
    resolve_arm_rho,
)
from blueberries_voi.experiments.modal_dispatch import run_batch
from blueberries_voi.experiments.policy_bakeoff_viz import (
    summarize_distribution_by_arm,
    write_alpha_vs_observed_figure,
    write_metric_boxplot_figure,
    write_paired_delta_boxplot_figure,
    write_paired_delta_figure,
    write_profit_bars_figure,
    write_runtime_bars_figure,
    write_runtime_violin_figure,
    write_waste_stockout_bars_figure,
)
from blueberries_voi.experiments.voi_profit import load_damped_sw_bo_params
from blueberries_voi.filter.types import channels_for_preset

BATCH_MODE: Literal["modal", "local"] = "modal"
SMOKE = False
SEEDS = (42,) if SMOKE else DEFAULT_CONTROLLER_SEEDS
N_BURN, N_SCORE = 2, PRODUCTION_N_SCORE
DEFAULT_RHO = load_damped_sw_bo_params()[1]
OUT_DIR = REPO_ROOT / "notebooks" / "outputs" / "nb21_controller_bakeoff"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OBS_CHANNELS = filtered_obs_channels()
ARM_RHOS = {arm: resolve_arm_rho(arm, default_rho=DEFAULT_RHO) for arm in FILTERED_ARMS}
ARM_ALPHAS = {arm: resolve_arm_alpha(arm) for arm in FILTERED_ARMS}
print(
    f"BATCH_MODE={BATCH_MODE} SMOKE={SMOKE} n_burn={N_BURN} n_score={N_SCORE} "
    f"horizon={N_BURN + N_SCORE} filtered_obs={FILTERED_OBS_PRESET} channels={OBS_CHANNELS} "
    f"rhos={ARM_RHOS} alphas={ARM_ALPHAS}"
)


## Modal bakeoff — filtered F3 (30 shards, flat spawn)

Filter posterior via `session.act`; F3 obs channels; 10 seeds × 3 arms.


In [ ]:
t0_filtered = time.perf_counter()
filtered_rows = run_batch(
    "controller_bakeoff",
    BATCH_MODE,
    smoke=SMOKE,
    seeds=SEEDS,
    arms=FILTERED_ARMS,
    rho=DEFAULT_RHO,
    belief_world="filtered",
    n_burn=N_BURN,
    n_score=N_SCORE,
    out_path=OUT_DIR / "filtered_f3_n45_rows.json",
)
filtered_wall_s = time.perf_counter() - t0_filtered
filtered_df = pd.DataFrame(filtered_rows)
print(f"Filtered Modal wall: {filtered_wall_s:.1f} s for {len(filtered_rows)} shards")
display(filtered_df.groupby("arm_id")[["alpha", "rho", "profit", "fill_rate", "day_no_stockout_rate"]].agg(["median", "mean"]).round(2))

shard_rows = list(filtered_rows)
(OUT_DIR / "controller_bakeoff_rows.json").write_text(json.dumps(shard_rows, indent=2) + "\n")
bakeoff_summary = {
    "n_burn": N_BURN,
    "n_score": N_SCORE,
    "horizon_days": N_BURN + N_SCORE,
    "filtered_obs": FILTERED_OBS_PRESET,
    "obs_channels": {
        "code_type": OBS_CHANNELS.code_type,
        "scan_waste": OBS_CHANNELS.scan_waste,
        "delivery_history": OBS_CHANNELS.delivery_history,
    },
    "arm_rhos": ARM_RHOS,
    "arm_alphas": ARM_ALPHAS,
    "filtered_modal_wall_s": filtered_wall_s,
    "filtered_shards": len(filtered_rows),
}
(OUT_DIR / "bakeoff_summary.json").write_text(json.dumps(bakeoff_summary, indent=2) + "\n")


## Per-seed tables + distribution summaries (45-day)


In [ ]:
df = pd.DataFrame(shard_rows)
per_seed = df.sort_values(["arm_id", "seed"])[
    ["seed", "arm_id", "alpha", "rho", "profit", "waste", "stockout", "fill_rate", "day_no_stockout_rate", "elapsed_s"]
]
display(per_seed.round(2).head(12))
per_seed.to_csv(OUT_DIR / "per_seed_filtered_n45.csv", index=False)

metrics = ["profit", "waste", "stockout", "fill_rate", "day_no_stockout_rate", "elapsed_s"]
for metric in metrics:
    tbl = summarize_distribution_by_arm(shard_rows, metric, arms=list(FILTERED_ARMS))
    tbl.to_csv(OUT_DIR / f"summary_filtered_{metric}_n45.csv")
    if metric == "profit":
        print("Filtered profit summary:")
        display(tbl.round(2))


## Distribution figures (45-day)


In [ ]:
write_profit_bars_figure(
    OUT_DIR / "01_profit_by_arm.png",
    shard_rows,
    title=f"Filtered {FILTERED_OBS_PRESET}: mean profit (45 scored days)",
)
write_metric_boxplot_figure(
    OUT_DIR / "06_profit_boxplot.png",
    shard_rows,
    "profit",
    ylabel="Profit ($)",
    title="Filtered profit distribution",
    arms=list(FILTERED_ARMS),
)
write_waste_stockout_bars_figure(OUT_DIR / "02_waste_stockout.png", shard_rows)
write_runtime_bars_figure(OUT_DIR / "03_runtime_by_arm.png", shard_rows)
write_runtime_violin_figure(OUT_DIR / "08_runtime_violin.png", shard_rows)
write_paired_delta_figure(
    OUT_DIR / "04_paired_delta_vs_sw.png",
    shard_rows,
    baseline="sw",
    title="Filtered F3: profit delta vs sw",
)
write_paired_delta_boxplot_figure(
    OUT_DIR / "09_paired_delta_boxplot_vs_sw.png",
    shard_rows,
    "profit",
    baseline="sw",
)
write_metric_boxplot_figure(
    OUT_DIR / "10_fill_rate.png",
    shard_rows,
    "fill_rate",
    ylabel="Fill rate",
    title="Filtered fill rate",
    arms=list(FILTERED_ARMS),
)
write_metric_boxplot_figure(
    OUT_DIR / "11_day_no_stockout.png",
    shard_rows,
    "day_no_stockout_rate",
    ylabel="Day no-stockout rate",
    title="Filtered day service",
    arms=list(FILTERED_ARMS),
)
write_alpha_vs_observed_figure(OUT_DIR / "12_alpha_vs_fill_rate.png", shard_rows, observed_field="fill_rate")
write_alpha_vs_observed_figure(
    OUT_DIR / "13_alpha_vs_day_no_stockout.png",
    shard_rows,
    observed_field="day_no_stockout_rate",
    title="Tuned α vs observed day no-stockout rate",
)
print(f"Wrote figures under {OUT_DIR}")


## Results summary (45 scored days, `n_burn=2`, horizon=47)

Modal wall time logged in `bakeoff_summary.json`.


In [ ]:
from IPython.display import Image, display

for fig_name in sorted(p.name for p in OUT_DIR.glob("*.png")):
    path = OUT_DIR / fig_name
    print(fig_name)
    display(Image(filename=str(path)))


## Analysis — filtered F3 (GSIN + scan waste + temperature history)

**Channel preset F3:** `code_type=gsin`, `scan_waste=true`, `delivery_history=temperature_history`.
Filtered posterior only; 10 seeds × 3 arms; 45 scored days after 2-day burn.

### Ranking (median profit)

1. **Damped SW** (~\$1,979) — best overall; ~\$340 above fixed order.
2. **Fixed order (constant)** (~\$1,639) — solid baseline with highest fill rate (~98.6%) and lowest stockouts (~17 units).
3. **Window SLA (PB)** (~\$203) — collapses under filtered F3: fill rate ~64%, stockouts ~443 units; tuned α≈0.56 and ρ=0.5 do not transfer.

### Constant vs adaptive

Under the richest practical obs package, **adaptive SW clearly beats fixed order** on profit while cutting waste sharply
(median ~58 vs ~430 units). The tradeoff is more stockouts (~65 vs ~17) but net economics favor SW.

**SLA-PB is not competitive** in this belief world — it under-orders relative to realized demand despite temperature history.

### Does richer data help controllers?

Yes for **SW**: F3 temperature traces + GSIN lot IDs give the filter enough structure for damped protection to
rebalance waste vs service profitably. Fixed order ignores the posterior entirely, so it cannot exploit that signal.
SLA-PB’s windowed path simulation appears miscalibrated on the filtered shelf (possibly α/ρ tuned on sparser channels).

### Runtime

Per-shard median wall time: constant ~1.4s, SW ~0.8s, SLA-PB ~0.5s (45-day horizon). Modal flat-spawn of 30 shards
completed in ~39s wall clock.


In [ ]:
med = filtered_df.groupby("arm_id")["profit"].median().reindex(list(FILTERED_ARMS))
mean = filtered_df.groupby("arm_id")["profit"].mean().reindex(list(FILTERED_ARMS))
rank = med.sort_values(ascending=False).index.tolist()
print("Median profit ranking:", rank)
print(med.round(1).to_string())

waste_med = filtered_df.groupby("arm_id")["waste"].median()
stock_med = filtered_df.groupby("arm_id")["stockout"].median()
fill_med = filtered_df.groupby("arm_id")["fill_rate"].median()
svc_med = filtered_df.groupby("arm_id")["day_no_stockout_rate"].median()
rt_med = filtered_df.groupby("arm_id")["elapsed_s"].median()
trade = pd.DataFrame({
    "median_profit": med,
    "median_waste": waste_med,
    "median_stockout": stock_med,
    "median_fill_rate": fill_med,
    "median_day_no_stockout": svc_med,
    "median_runtime_s": rt_med,
}).round(2)
display(trade)

delta_vs_constant = (med - med["constant"]).round(1)
delta_vs_sw = (med - med["sw"]).round(1)
print("Δ median profit vs constant:", delta_vs_constant.to_dict())
print("Δ median profit vs sw:", delta_vs_sw.to_dict())

constant_wins = rank[0] == "constant"
print(f"Fixed-order constant wins? {constant_wins}")
if not constant_wins:
    best = rank[0]
    print(f"Best adaptive arm {best} beats constant by ${med[best] - med['constant']:.1f} median profit.")
else:
    gap = med["constant"] - med[rank[1]]
    print(f"Constant leads {rank[1]} by ${gap:.1f}; richer F3 data did not overturn fixed order.")
